# Stochastic Biophysics Practice: From NumPy to Gillespie SSA

이 노트북은 확률론적 생물물리학 입문 실습을 위한 Colab-ready 자료입니다.
외부 데이터 없이 NumPy와 Matplotlib만 사용하며, 위에서 아래로 순서대로 실행하면 됩니다.

오늘 실습에서 다룰 것:

- NumPy 배열과 난수
- Matplotlib 기초 시각화
- 평균, 분산, 표준편차, histogram
- Bernoulli, Binomial, Poisson, Gaussian
- Random walk와 diffusion
- Markov chain
- Poisson process와 waiting time
- Gillespie stochastic simulation algorithm
- Birth-death model과 간단한 화학 반응식 예시
- Time-delayed birth-death process와 생산 지연(production delay)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

print("NumPy version:", np.__version__)
print("Random seed: 2026")

## 1. NumPy and Matplotlib basics

NumPy 배열은 여러 값을 한 번에 계산하게 해 줍니다. 아래 셀에서는 배열 생성, 벡터화 연산, 난수 생성,
그리고 `plot`, `scatter`, `hist`, `step` 시각화를 빠르게 확인합니다.

In [ ]:
values = np.array([1, 2, 3, 4, 5])
squared = values**2
grid = np.linspace(0, 2 * np.pi, 200)
sine = np.sin(grid)
noisy_sine = sine + 0.25 * rng.normal(size=grid.size)
poisson_counts = rng.poisson(lam=4.0, size=12)
channel_states = rng.choice(["closed", "open"], size=12, p=[0.7, 0.3])

print("values:", values)
print("values**2:", squared)
print("Poisson counts:", poisson_counts)
print("Channel states:", channel_states)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(grid, sine, label="sin(t)")
axes[0, 0].set(title="Line Plot", xlabel="t", ylabel="sin(t)")
axes[0, 0].legend()

axes[0, 1].scatter(grid[::4], noisy_sine[::4], s=15, label="noisy samples")
axes[0, 1].set(title="Scatter Plot", xlabel="t", ylabel="value")
axes[0, 1].legend()

axes[1, 0].hist(noisy_sine, bins=25, alpha=0.8, label="noisy sine")
axes[1, 0].set(title="Histogram", xlabel="value", ylabel="count")
axes[1, 0].legend()

axes[1, 1].step(np.arange(poisson_counts.size), poisson_counts, where="mid", label="counts")
axes[1, 1].set(title="Step Plot", xlabel="index", ylabel="count")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

**Try it**

- `rng.normal`의 표준편차를 바꿔서 scatter plot이 어떻게 달라지는지 확인해 보세요.
- `rng.poisson(lam=4.0)`에서 `lam`을 1, 10으로 바꿔 count의 변화를 비교해 보세요.

## 2. Basic statistics and visualization

표본 평균, 분산, 표준편차는 확률적 실험 결과를 요약하는 가장 기본적인 도구입니다.
표본 수가 커질수록 평균 추정이 안정되는 것과, Poisson counting noise의 상대 변동이 작아지는 것을 확인합니다.

In [ ]:
def summarize_samples(x):
    """Return sample size, mean, variance, and standard deviation."""
    x = np.asarray(x)
    return {
        "n": int(x.size),
        "mean": float(np.mean(x)),
        "variance": float(np.var(x, ddof=1)),
        "std": float(np.std(x, ddof=1)),
    }


normal_samples = rng.normal(loc=2.0, scale=1.5, size=5000)
print("Summary for normal samples:", summarize_samples(normal_samples))

sample_sizes = np.array([10, 30, 100, 300, 1000, 3000, 10000])
mean_estimates = np.array([
    np.mean(rng.normal(loc=2.0, scale=1.5, size=n)) for n in sample_sizes
])

poisson_means = np.array([2, 5, 10, 20, 50, 100])
relative_fluctuation = []
for mu in poisson_means:
    counts = rng.poisson(lam=mu, size=20000)
    relative_fluctuation.append(np.std(counts) / np.mean(counts))
relative_fluctuation = np.array(relative_fluctuation)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(normal_samples, bins=40, alpha=0.8, label="samples")
axes[0].axvline(np.mean(normal_samples), color="black", linestyle="--", label="mean")
axes[0].set(title="Empirical Distribution", xlabel="value", ylabel="count")
axes[0].legend()

axes[1].plot(sample_sizes, mean_estimates, marker="o", label="sample mean")
axes[1].axhline(2.0, color="black", linestyle="--", label="true mean")
axes[1].set_xscale("log")
axes[1].set(title="Mean Estimate vs Sample Size", xlabel="sample size", ylabel="mean")
axes[1].legend()

axes[2].plot(poisson_means, relative_fluctuation, marker="o", label="simulation")
axes[2].plot(poisson_means, 1 / np.sqrt(poisson_means), marker="s", label="1/sqrt(mean)")
axes[2].set(title="Counting Noise", xlabel="mean count", ylabel="std / mean")
axes[2].legend()

plt.tight_layout()
plt.show()

**Exercise**

- `normal_samples`의 `scale`을 바꾸면 histogram과 표준편차가 어떻게 변하는지 확인해 보세요.
- Poisson 평균이 커질 때 `std / mean`이 왜 작아지는지 그래프를 보고 설명해 보세요.

## 3. Bernoulli, Binomial, Poisson, Gaussian

이 섹션에서는 가장 자주 만나는 확률분포를 simulation histogram과 이론 PMF/PDF로 비교합니다.
`scipy` 없이 `math.lgamma`를 사용해 factorial이 들어가는 PMF를 안정적으로 계산합니다.

In [ ]:
def binomial_pmf(k_values, n, p):
    """Compute Binomial(n, p) PMF for integer k values without scipy."""
    k_values = np.asarray(k_values, dtype=int)
    probs = []
    for k in k_values:
        if k < 0 or k > n:
            probs.append(0.0)
            continue
        if p == 0:
            probs.append(1.0 if k == 0 else 0.0)
            continue
        if p == 1:
            probs.append(1.0 if k == n else 0.0)
            continue
        log_prob = (
            math.lgamma(n + 1)
            - math.lgamma(k + 1)
            - math.lgamma(n - k + 1)
            + k * math.log(p)
            + (n - k) * math.log(1 - p)
        )
        probs.append(math.exp(log_prob))
    return np.array(probs)


def poisson_pmf(k_values, lam):
    """Compute Poisson(lam) PMF for integer k values without scipy."""
    k_values = np.asarray(k_values, dtype=int)
    probs = []
    for k in k_values:
        if k < 0:
            probs.append(0.0)
            continue
        log_prob = k * math.log(lam) - lam - math.lgamma(k + 1)
        probs.append(math.exp(log_prob))
    return np.array(probs)


def gaussian_pdf(x, mu, sigma):
    """Return Gaussian probability density."""
    x = np.asarray(x)
    z = (x - mu) / sigma
    return np.exp(-0.5 * z**2) / (sigma * np.sqrt(2 * np.pi))

In [ ]:
bernoulli_samples = rng.choice([0, 1], size=1000, p=[0.65, 0.35])

n_channels = 20
p_open = 0.3
open_counts = rng.binomial(n=n_channels, p=p_open, size=10000)
k_binom = np.arange(0, n_channels + 1)
binom_theory = binomial_pmf(k_binom, n_channels, p_open)

lam = 4.0
rare_event_counts = rng.poisson(lam=lam, size=10000)
k_pois = np.arange(0, 16)
poisson_theory = poisson_pmf(k_pois, lam)

limit_counts = rng.binomial(n=100, p=0.04, size=10000)
k_limit = np.arange(0, 16)
limit_theory = poisson_pmf(k_limit, 100 * 0.04)

fano = np.var(rare_event_counts, ddof=1) / np.mean(rare_event_counts)
print(f"Bernoulli open fraction: {np.mean(bernoulli_samples):.3f}")
print(f"Poisson empirical mean: {np.mean(rare_event_counts):.3f}")
print(f"Poisson empirical variance: {np.var(rare_event_counts, ddof=1):.3f}")
print(f"Poisson Fano factor Var/Mean: {fano:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].bar([0, 1], [np.mean(bernoulli_samples == 0), np.mean(bernoulli_samples == 1)])
axes[0, 0].set(title="Bernoulli Trial", xlabel="state", ylabel="frequency")
axes[0, 0].set_xticks([0, 1], ["closed", "open"])

axes[0, 1].hist(open_counts, bins=np.arange(-0.5, n_channels + 1.5), density=True, alpha=0.7, label="simulation")
axes[0, 1].plot(k_binom, binom_theory, "o-", label="Binomial PMF")
axes[0, 1].set(title="Open Ion Channels", xlabel="open channels", ylabel="probability")
axes[0, 1].legend()

axes[1, 0].hist(rare_event_counts, bins=np.arange(-0.5, 16.5), density=True, alpha=0.7, label="simulation")
axes[1, 0].plot(k_pois, poisson_theory, "o-", label="Poisson PMF")
axes[1, 0].set(title="Rare Event Counts", xlabel="count", ylabel="probability")
axes[1, 0].legend()

axes[1, 1].hist(limit_counts, bins=np.arange(-0.5, 16.5), density=True, alpha=0.7, label="Binomial simulation")
axes[1, 1].plot(k_limit, limit_theory, "o-", label="Poisson PMF")
axes[1, 1].set(title="Poisson Limit of Binomial", xlabel="count", ylabel="probability")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for lam_value in [3, 20, 80]:
    k = np.arange(0, int(lam_value + 5 * np.sqrt(lam_value)) + 1)
    pmf = poisson_pmf(k, lam_value)
    axes[0].plot(k, pmf, "o-", markersize=3, label=f"Poisson mean={lam_value}")
    x_dense = np.linspace(k.min(), k.max(), 300)
    axes[0].plot(x_dense, gaussian_pdf(x_dense, lam_value, np.sqrt(lam_value)), "--", label=f"Gaussian mean={lam_value}")

sigma = 2.0
mu = 0.0
x = np.linspace(-8, 8, 400)
pdf = gaussian_pdf(x, mu, sigma)
fwhm = 2 * math.sqrt(2 * math.log(2)) * sigma
half_max = pdf.max() / 2

axes[1].plot(x, pdf, label="Gaussian PDF")
axes[1].axhline(half_max, color="black", linestyle="--", label="half maximum")
axes[1].axvline(mu - fwhm / 2, color="red", linestyle=":", label="FWHM edges")
axes[1].axvline(mu + fwhm / 2, color="red", linestyle=":")
axes[1].set(title="Gaussian FWHM", xlabel="x", ylabel="density")
axes[1].legend()

axes[0].set(title="Gaussian Approximation", xlabel="count", ylabel="probability")
axes[0].legend(fontsize=8)

print(f"FWHM = {fwhm:.3f}, 2.355 sigma = {2.355 * sigma:.3f}")

plt.tight_layout()
plt.show()

**Exercise**

- `n_channels`와 `p_open`을 바꿔 Binomial histogram의 폭과 평균을 비교해 보세요.
- Poisson 평균이 커질 때 Gaussian 근사가 왜 더 좋아지는지 그래프에서 확인해 보세요.

## 4. Random walk and diffusion

1D unbiased random walk에서는 ensemble 평균이 0 근처에 머물고, 분산은 시간에 비례해 증가합니다.
step length를 `L`, time step을 `dt`라고 하면 `L^2 = 2Ddt`로 diffusion coefficient `D`를 연결할 수 있습니다.

In [ ]:
def simulate_random_walk(n_trajectories, n_steps, p_plus=0.5, step_length=1.0, rng=None):
    """Simulate 1D random walks with steps +/- step_length."""
    if rng is None:
        rng = np.random.default_rng()
    steps = rng.choice(
        [-step_length, step_length],
        size=(n_trajectories, n_steps),
        p=[1 - p_plus, p_plus],
    )
    positions = np.concatenate(
        [np.zeros((n_trajectories, 1)), np.cumsum(steps, axis=1)],
        axis=1,
    )
    return positions


n_trajectories = 2000
n_steps = 200
step_length = 0.2
dt = 0.01
diffusion_coefficient = step_length**2 / (2 * dt)
time = np.arange(n_steps + 1) * dt

walks = simulate_random_walk(n_trajectories, n_steps, step_length=step_length, rng=rng)
drift_walks = simulate_random_walk(n_trajectories, n_steps, p_plus=0.56, step_length=step_length, rng=rng)

mean_position = np.mean(walks, axis=0)
variance_position = np.var(walks, axis=0)
drift_mean = np.mean(drift_walks, axis=0)

print(f"D from L^2 = 2Ddt: {diffusion_coefficient:.3f}")
print(f"Final unbiased mean: {mean_position[-1]:.3f}")
print(f"Final unbiased variance: {variance_position[-1]:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(time, walks[:8].T, alpha=0.8)
axes[0].set(title="Random Walk Trajectories", xlabel="time", ylabel="position")

axes[1].plot(time, variance_position, label="simulation")
axes[1].plot(time, 2 * diffusion_coefficient * time, "--", label="2Dt")
axes[1].set(title="Variance vs Time", xlabel="time", ylabel="variance")
axes[1].legend()

axes[2].hist(walks[:, -1], bins=35, alpha=0.7, label="unbiased final")
axes[2].axvline(np.mean(walks[:, -1]), color="black", linestyle="--", label="mean")
axes[2].plot(time, drift_mean, color="red", label="drift mean")
axes[2].set(title="Final Positions and Drift Mean", xlabel="position or time", ylabel="count / position")
axes[2].legend()

plt.tight_layout()
plt.show()

**Try it**

- `p_plus`를 0.5보다 크게 또는 작게 바꿔 평균 위치가 어떤 방향으로 이동하는지 확인해 보세요.
- `step_length`와 `dt`를 바꾸고 `variance vs time` 기울기가 어떻게 변하는지 비교해 보세요.

## 5. Markov chain

Discrete-time Markov chain에서는 현재 state와 transition matrix가 다음 state의 확률을 정합니다.
여기서는 2-state ion channel과 3-state cyclic model을 다룹니다.

In [ ]:
def simulate_markov_chain(P, initial_state, n_steps, rng):
    """Simulate a discrete-time Markov chain with row-stochastic matrix P."""
    n_states = P.shape[0]
    states = np.empty(n_steps + 1, dtype=int)
    states[0] = initial_state
    for step in range(n_steps):
        states[step + 1] = rng.choice(n_states, p=P[states[step]])
    return states


P = np.array([
    [0.92, 0.08],
    [0.25, 0.75],
])
state_names = np.array(["Closed", "Open"])

probabilities = [np.array([1.0, 0.0])]
for _ in range(80):
    probabilities.append(probabilities[-1] @ P)
probabilities = np.array(probabilities)

stationary = np.array([1.0, 0.0])
for _ in range(1000):
    stationary = stationary @ P

channel_trajectory = simulate_markov_chain(P, initial_state=0, n_steps=80, rng=rng)
print("Approximate stationary distribution:", dict(zip(state_names, stationary.round(3))))

P3 = np.array([
    [0.05, 0.95, 0.00],
    [0.00, 0.10, 0.90],
    [0.85, 0.00, 0.15],
])
prob3 = [np.array([1.0, 0.0, 0.0])]
for _ in range(80):
    prob3.append(prob3[-1] @ P3)
prob3 = np.array(prob3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].step(np.arange(channel_trajectory.size), channel_trajectory, where="post", label="state")
axes[0].set(title="Ion Channel Trajectory", xlabel="step", ylabel="state")
axes[0].set_yticks([0, 1], state_names)
axes[0].legend()

axes[1].plot(probabilities[:, 0], label="Closed")
axes[1].plot(probabilities[:, 1], label="Open")
axes[1].set(title="State Probability", xlabel="step", ylabel="probability")
axes[1].legend()

for i, name in enumerate(["A", "B", "C"]):
    axes[2].plot(prob3[:, i], label=name)
axes[2].set(title="Three-State Cycle", xlabel="step", ylabel="probability")
axes[2].legend()

plt.tight_layout()
plt.show()

**Exercise**

- Transition matrix `P`에서 closed-to-open 확률을 키우면 stationary open probability가 어떻게 바뀌는지 확인해 보세요.
- `P3`의 순환 확률을 낮추면 세 state의 probability curve가 어떻게 달라지는지 비교해 보세요.

## 6. Poisson process and waiting time

일정 rate `beta`의 Poisson process는 작은 시간 간격에서 Bernoulli event를 반복하는 방식으로 근사할 수 있고,
exponential waiting time을 직접 샘플링해서도 만들 수 있습니다.

In [ ]:
beta = 3.0
total_time = 5.0
dt_small = 0.001
n_replicates = 5000

n_small_steps = int(total_time / dt_small)
counts_from_small_dt = rng.binomial(n=n_small_steps, p=beta * dt_small, size=n_replicates)
counts_direct = rng.poisson(lam=beta * total_time, size=n_replicates)
waiting_times = rng.exponential(scale=1 / beta, size=10000)

event_times = []
current_time = 0.0
while current_time < total_time:
    current_time += rng.exponential(scale=1 / beta)
    if current_time < total_time:
        event_times.append(current_time)
event_times = np.array(event_times)

print(f"Mean count from small-dt approximation: {np.mean(counts_from_small_dt):.3f}")
print(f"Mean count from direct Poisson sampling: {np.mean(counts_direct):.3f}")
print(f"Expected count beta*T: {beta * total_time:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x_wait = np.linspace(0, waiting_times.max(), 300)
axes[0].hist(waiting_times, bins=50, density=True, alpha=0.7, label="simulation")
axes[0].plot(x_wait, beta * np.exp(-beta * x_wait), label="beta exp(-beta tau)")
axes[0].set(title="Waiting Time Distribution", xlabel="tau", ylabel="density")
axes[0].legend()

k_count = np.arange(0, int(beta * total_time + 6 * np.sqrt(beta * total_time)) + 1)
axes[1].hist(counts_from_small_dt, bins=np.arange(-0.5, k_count.max() + 1.5), density=True, alpha=0.6, label="small dt")
axes[1].plot(k_count, poisson_pmf(k_count, beta * total_time), "o-", label="Poisson PMF")
axes[1].set(title="Event Count in Fixed Time", xlabel="count", ylabel="probability")
axes[1].legend()

event_grid = np.r_[0.0, event_times, total_time]
cumulative_count = np.r_[0, np.arange(1, event_times.size + 1), event_times.size]
axes[2].step(event_grid, cumulative_count, where="post", label="single process")
axes[2].set(title="Poisson Process Path", xlabel="time", ylabel="event count")
axes[2].legend()

plt.tight_layout()
plt.show()

**Try it**

- `beta`를 키우면 waiting time histogram과 event count histogram이 어떻게 바뀌는지 확인해 보세요.
- `dt_small`을 너무 크게 잡으면 small-dt approximation이 왜 나빠질 수 있는지 생각해 보세요.

## 7. Gillespie stochastic simulation algorithm

Gillespie SSA는 continuous-time Markov jump process를 정확하게 샘플링하는 알고리즘입니다.
현재 state에서 각 반응의 propensity를 계산하고, 전체 rate로 다음 반응 시간과 반응 종류를 고릅니다.

In [ ]:
def simulate_gillespie(x0, reactions, propensities, t_max, rng, max_steps=100000):
    """Simulate a reaction network with Gillespie stochastic simulation algorithm."""
    x = np.array(x0, dtype=int)
    reactions = np.array(reactions, dtype=int)
    t = 0.0

    times = [t]
    states = [x.copy()]
    reaction_counts = np.zeros(len(reactions), dtype=int)

    for _ in range(max_steps):
        rates = np.asarray(propensities(x), dtype=float)
        if np.any(rates < -1e-12):
            raise ValueError("Propensities must be non-negative.")
        rates = np.maximum(rates, 0.0)
        a0 = np.sum(rates)

        if a0 <= 0 or t >= t_max:
            break

        r1 = max(rng.random(), np.finfo(float).tiny)
        tau = -math.log(r1) / a0
        if t + tau > t_max:
            break

        threshold = rng.random() * a0
        reaction_index = int(np.searchsorted(np.cumsum(rates), threshold, side="right"))
        x_next = x + reactions[reaction_index]

        if np.any(x_next < 0):
            raise ValueError("A reaction produced a negative molecule count.")

        t += tau
        x = x_next
        times.append(t)
        states.append(x.copy())
        reaction_counts[reaction_index] += 1

    return np.array(times), np.vstack(states), reaction_counts


def sample_step_trajectory(times, values, grid):
    """Sample a piecewise-constant trajectory on a regular time grid."""
    idx = np.searchsorted(times, grid, side="right") - 1
    idx = np.clip(idx, 0, len(values) - 1)
    return values[idx]


print("Gillespie SSA helper functions are ready.")

**Exercise**

- `simulate_gillespie`에서 `a0 <= 0`일 때 왜 simulation을 멈추는지 설명해 보세요.
- `max_steps`가 필요한 이유를 생각해 보세요.

## 8. Birth-death model

Birth-death model은 `∅ -> X`와 `X -> ∅` 두 반응으로 이루어진 가장 간단한 stochastic production-degradation model입니다.
Ensemble mean은 결정론적 ODE 해와 비교할 수 있고, stationary distribution은 Poisson에 가까워집니다.

In [ ]:
beta_birth = 12.0
k_decay = 0.6
x0_birth_death = np.array([0])
reactions_birth_death = np.array([[1], [-1]])
t_max_birth_death = 20.0


def birth_death_propensities(x):
    return np.array([beta_birth, k_decay * x[0]])


grid_birth_death = np.linspace(0, t_max_birth_death, 201)
example_trajectories = []
for _ in range(6):
    times, states, _ = simulate_gillespie(
        x0_birth_death,
        reactions_birth_death,
        birth_death_propensities,
        t_max_birth_death,
        rng,
    )
    example_trajectories.append(sample_step_trajectory(times, states[:, 0], grid_birth_death))
example_trajectories = np.array(example_trajectories)

n_ensemble = 300
ensemble = np.zeros((n_ensemble, grid_birth_death.size))
for i in range(n_ensemble):
    times, states, _ = simulate_gillespie(
        x0_birth_death,
        reactions_birth_death,
        birth_death_propensities,
        t_max_birth_death,
        rng,
    )
    ensemble[i] = sample_step_trajectory(times, states[:, 0], grid_birth_death)

ensemble_mean = np.mean(ensemble, axis=0)
ode_mean = beta_birth / k_decay + (x0_birth_death[0] - beta_birth / k_decay) * np.exp(-k_decay * grid_birth_death)
stationary_samples = ensemble[:, -1].astype(int)
stationary_mean = np.mean(stationary_samples)
stationary_variance = np.var(stationary_samples, ddof=1)
stationary_fano = stationary_variance / stationary_mean

print(f"Expected stationary mean beta/k: {beta_birth / k_decay:.3f}")
print(f"Empirical stationary mean: {stationary_mean:.3f}")
print(f"Empirical stationary variance: {stationary_variance:.3f}")
print(f"Empirical Fano factor: {stationary_fano:.3f}")

k_stationary = np.arange(0, max(stationary_samples.max(), int(beta_birth / k_decay + 6 * np.sqrt(beta_birth / k_decay))) + 1)
stationary_theory = poisson_pmf(k_stationary, beta_birth / k_decay)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].step(grid_birth_death, example_trajectories.T, where="post", alpha=0.8)
axes[0].set(title="Birth-Death Trajectories", xlabel="time", ylabel="X")

axes[1].plot(grid_birth_death, ensemble_mean, label="ensemble mean")
axes[1].plot(grid_birth_death, ode_mean, "--", label="ODE mean")
axes[1].set(title="Mean vs ODE", xlabel="time", ylabel="mean X")
axes[1].legend()

axes[2].hist(stationary_samples, bins=np.arange(-0.5, k_stationary.max() + 1.5), density=True, alpha=0.7, label="simulation")
axes[2].plot(k_stationary, stationary_theory, "o-", markersize=3, label="Poisson PMF")
axes[2].set(title="Stationary Distribution", xlabel="X", ylabel="probability")
axes[2].legend()

plt.tight_layout()
plt.show()

**Try it**

- `beta_birth`를 키우면 stationary mean과 noise의 상대 크기(`std / mean`)가 어떻게 변하는지 확인해 보세요.
- `k_decay`를 키우면 steady state가 어떻게 이동하는지 ODE curve와 함께 비교해 보세요.

## 9. Time-delayed birth-death process

실제 유전자 발현에서는 전사와 번역에 시간이 걸립니다. 생산 지연(production delay) τ를 도입한 birth-death model에서는 birth event가 발화할 때 분자가 즉시 추가되는 대신, τ 시간 후에 추가됩니다.

- **Production**: ∅ →^β X (birth fires immediately, but X appears after delay τ)
- **Degradation**: X →^k ∅ (instantaneous)

지연 Gillespie 알고리즘은 표준 SSA에 **scheduled-event queue(min-heap)** 를 추가합니다.
- Birth event 발화 시 완료 시각 t + τ를 큐에 추가 (X는 즉시 증가하지 않음)
- Death event는 현재 성숙한(mature) 분자 X에만 즉시 작용

**Key predictions**:
- Stationary mean ⟨X⟩ = β/k (지연이 없는 경우와 동일)
- Stationary Fano factor > 1 — 지연이 커질수록 Poisson보다 noise 증가
- 빈 상태(x₀ = 0)에서 출발하면 분자는 최소 τ 이후에야 처음 나타남

In [ ]:
import heapq


def simulate_delayed_birth_death(beta, k, tau_delay, x0, t_max, rng, max_steps=500_000):
    """Gillespie SSA for birth-death with fixed production delay.

    Birth fires at rate beta but the new molecule appears tau_delay later.
    Death acts only on currently mature molecules at rate k*x.
    """
    t = 0.0
    x = int(x0)
    times = [t]
    values = [x]
    pending = []  # min-heap of scheduled birth-completion times

    for _ in range(max_steps):
        t_pending = pending[0] if pending else np.inf
        a0 = beta + k * x

        if a0 > 0:
            dt = -math.log(max(rng.random(), np.finfo(float).tiny)) / a0
            t_reaction = t + dt
        else:
            t_reaction = np.inf

        if t_pending <= t_reaction:
            if t_pending > t_max:
                break
            t = heapq.heappop(pending)
            x += 1
            times.append(t)
            values.append(x)
        else:
            if t_reaction > t_max:
                break
            t = t_reaction
            if rng.random() * a0 < beta:
                heapq.heappush(pending, t + tau_delay)
            else:
                x = max(x - 1, 0)
                times.append(t)
                values.append(x)

    if times[-1] < t_max:
        times.append(t_max)
        values.append(values[-1])
    return np.array(times), np.array(values)


def delayed_birth_death_ode(grid, beta, k, tau, x0):
    """Euler integration of the DDE mean for constant-rate birth with delay."""
    y = np.empty_like(grid, dtype=float)
    y[0] = float(x0)
    for i in range(1, grid.size):
        dt = grid[i] - grid[i - 1]
        beta_eff = beta if (grid[i - 1] - tau) >= 0.0 else 0.0
        y[i] = max(0.0, y[i - 1] + dt * (beta_eff - k * y[i - 1]))
    return y


delay_beta = 12.0
delay_k = 0.6
tau_values = [0.0, 0.5, 2.0]
delay_t_max = 25.0
delay_grid = np.linspace(0, delay_t_max, 251)
n_delay_ensemble = 300

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# panel 1: ensemble mean vs DDE for each tau
for tau in tau_values:
    ensemble = np.zeros((n_delay_ensemble, delay_grid.size))
    for i in range(n_delay_ensemble):
        t_, v_ = simulate_delayed_birth_death(delay_beta, delay_k, tau, 0, delay_t_max, rng)
        ensemble[i] = sample_step_trajectory(t_, v_, delay_grid)
    mean_curve = np.mean(ensemble, axis=0)
    ode_curve = delayed_birth_death_ode(delay_grid, delay_beta, delay_k, tau, 0)
    axes[0].plot(delay_grid, mean_curve, label=f"\u03c4={tau} (sim)")
    axes[0].plot(delay_grid, ode_curve, "--", alpha=0.5)
    fano = np.var(ensemble[:, -1].astype(float), ddof=1) / np.mean(ensemble[:, -1])
    print(f"tau={tau:.1f}  stationary mean={np.mean(ensemble[:, -1]):.2f}  Fano={fano:.3f}")

axes[0].axhline(delay_beta / delay_k, color="black", linestyle=":", label="\u03b2/k")
axes[0].set(title="Ensemble Mean vs DDE (dashed)", xlabel="time", ylabel="mean X")
axes[0].legend(fontsize=8)

# panel 2: individual trajectories for tau=2.0
for _ in range(6):
    t_, v_ = simulate_delayed_birth_death(delay_beta, delay_k, 2.0, 0, delay_t_max, rng)
    axes[1].step(t_, v_, where="post", alpha=0.75)
axes[1].axhline(delay_beta / delay_k, color="black", linestyle=":", label="\u03b2/k")
axes[1].axvline(2.0, color="red", linestyle="--", alpha=0.7, label="t = \u03c4")
axes[1].set(title="Sample Trajectories (\u03c4 = 2)", xlabel="time", ylabel="X")
axes[1].legend()

# panel 3: Fano factor vs tau
tau_scan = np.linspace(0.0, 4.0, 20)
fano_scan = []
for tau in tau_scan:
    samp = np.zeros(n_delay_ensemble)
    for i in range(n_delay_ensemble):
        t_, v_ = simulate_delayed_birth_death(delay_beta, delay_k, tau, 0, delay_t_max, rng)
        samp[i] = v_[-1]
    mu = np.mean(samp)
    fano_scan.append(np.var(samp, ddof=1) / mu if mu > 0 else np.nan)

axes[2].plot(tau_scan, fano_scan, "o-", label="simulation")
axes[2].axhline(1.0, color="black", linestyle="--", label="Poisson (Fano = 1)")
axes[2].set(title="Fano Factor vs Delay \u03c4", xlabel="\u03c4", ylabel="Var(X) / Mean(X)")
axes[2].legend()

plt.tight_layout()
plt.show()

**Exercise**

- `tau_delay`를 0에서 키우면 ensemble mean이 처음 올라오는 시점이 어떻게 달라지는지 확인해 보세요.
- Fano factor가 τ = 0일 때 ≈ 1이고 τ가 커질수록 증가하는 이유를 직관적으로 설명해 보세요.
- 정상 상태에서 "in transit" 중인 분자의 평균 수를 β와 τ로 나타내 보세요. (힌트: Little's law)

## 10. Chemical reaction examples with Gillespie

같은 Gillespie 함수를 사용해 더 일반적인 반응 네트워크도 simulation할 수 있습니다.
여기서는 reversible conversion과 dimerization-like reaction을 교육용으로 단순하게 구현합니다.

In [ ]:
k1 = 0.25
k2 = 0.08
reactions_conversion = np.array([[-1, 1], [1, -1]])


def conversion_propensities(x):
    A, B = x
    return np.array([k1 * A, k2 * B])


times_conv, states_conv, counts_conv = simulate_gillespie(
    x0=np.array([40, 0]),
    reactions=reactions_conversion,
    propensities=conversion_propensities,
    t_max=40.0,
    rng=rng,
)
total_molecules = np.sum(states_conv, axis=1)
print("Reversible conversion event counts:", counts_conv)
print("A+B conserved:", np.all(total_molecules == total_molecules[0]))
print("Minimum molecule count:", states_conv.min())

k_forward = 0.002
k_reverse = 0.15
reactions_dimer = np.array([[-2, 1], [2, -1]])


def dimer_propensities(x):
    A, B = x
    return np.array([
        k_forward * A * (A - 1) / 2,
        k_reverse * B,
    ])


times_dim, states_dim, counts_dim = simulate_gillespie(
    x0=np.array([60, 0]),
    reactions=reactions_dimer,
    propensities=dimer_propensities,
    t_max=80.0,
    rng=rng,
)
conserved_units = states_dim[:, 0] + 2 * states_dim[:, 1]
print("Dimerization-like event counts:", counts_dim)
print("A+2B conserved:", np.all(conserved_units == conserved_units[0]))
print("Minimum molecule count:", states_dim.min())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].step(times_conv, states_conv[:, 0], where="post", label="A")
axes[0].step(times_conv, states_conv[:, 1], where="post", label="B")
axes[0].set(title="Reversible Conversion", xlabel="time", ylabel="molecule count")
axes[0].legend()

axes[1].step(times_dim, states_dim[:, 0], where="post", label="A")
axes[1].step(times_dim, states_dim[:, 1], where="post", label="B")
axes[1].set(title="Dimerization-Like Reaction", xlabel="time", ylabel="molecule count")
axes[1].legend()

plt.tight_layout()
plt.show()

**Exercise**

- Reversible conversion에서 `k1`과 `k2`의 비율을 바꾸면 A와 B의 장기적인 비율이 어떻게 변하는지 확인해 보세요.
- Dimerization-like reaction에서 `k_forward`를 키우면 B가 더 많이 생기는지 trajectory로 확인해 보세요.

## 11. Mini project

아래 파라미터를 직접 바꿔 보면서 stochastic model과 deterministic ODE의 차이를 관찰합니다.

질문:

- `beta`를 키우면 평균과 noise는 어떻게 변하는가?
- `k`를 키우면 steady state는 어떻게 변하는가?
- birth-death model의 stationary histogram은 언제 Poisson과 잘 맞는가?
- deterministic ODE와 stochastic trajectory는 언제 비슷해지는가?

In [ ]:
project_beta = 8.0
project_k = 0.4
project_x0 = 5
project_t_max = 25.0
project_replicates = 200


def run_birth_death_project(beta, k, x0, t_max, n_replicates, rng):
    """Run a compact birth-death experiment for parameter exploration."""
    reactions = np.array([[1], [-1]])

    def propensities(x):
        return np.array([beta, k * x[0]])

    grid = np.linspace(0, t_max, 200)
    ensemble = np.zeros((n_replicates, grid.size))

    for i in range(n_replicates):
        times, states, _ = simulate_gillespie(np.array([x0]), reactions, propensities, t_max, rng)
        ensemble[i] = sample_step_trajectory(times, states[:, 0], grid)

    mean_curve = np.mean(ensemble, axis=0)
    ode_curve = beta / k + (x0 - beta / k) * np.exp(-k * grid)
    final_values = ensemble[:, -1].astype(int)
    fano_factor = np.var(final_values, ddof=1) / np.mean(final_values)

    print(f"beta={beta}, k={k}, x0={x0}, t_max={t_max}")
    print(f"Expected steady state beta/k: {beta / k:.3f}")
    print(f"Final mean: {np.mean(final_values):.3f}")
    print(f"Final variance: {np.var(final_values, ddof=1):.3f}")
    print(f"Final Fano factor: {fano_factor:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].step(grid, ensemble[:8].T, where="post", alpha=0.7)
    axes[0].plot(grid, ode_curve, color="black", linewidth=2, label="ODE mean")
    axes[0].set(title="Mini Project Trajectories", xlabel="time", ylabel="X")
    axes[0].legend()

    axes[1].plot(grid, mean_curve, label="ensemble mean")
    axes[1].plot(grid, ode_curve, "--", label="ODE mean")
    axes[1].set(title="Mini Project Mean", xlabel="time", ylabel="mean X")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return final_values


project_final_values = run_birth_death_project(
    project_beta,
    project_k,
    project_x0,
    project_t_max,
    project_replicates,
    rng,
)

**Exercise**

- `project_beta`, `project_k`, `project_x0`, `project_t_max`를 하나씩 바꿔 결과를 비교해 보세요.
- 같은 평균 `beta / k`를 유지하면서 `beta`와 `k`를 동시에 키우면 trajectory가 어떻게 달라지는지 확인해 보세요.

## 12. Stochastic paths, mean field, heatmap, and Allee effect

Allee effect가 있는 population model에서는 작은 population이 오히려 감소하는 방향으로 drift할 수 있습니다.
아래 mean-field 식은 세 equilibrium을 가집니다: extinction state `N = 0`, unstable Allee threshold `N = A`, stable carrying capacity `N = K`.

\[
\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right)
\]

stochastic model에서는 같은 mean-field drift를 birth-death propensity의 차이로 구현합니다. 따라서 threshold 근처에서는 같은 초기값에서도 어떤 replicate는 extinction으로, 어떤 replicate는 survival로 갈 수 있습니다.

In [ ]:
def allee_drift(n, r, K, A):
    """Mean-field drift for a strong Allee effect model."""
    n = np.asarray(n, dtype=float)
    return r * n * (1 - n / K) * (n / A - 1)


def allee_birth_death_rates(n, r, K, A, turnover=0.2, max_population=None):
    """Return birth and death rates whose difference equals the Allee drift."""
    n = int(n)
    if n <= 0:
        return 0.0, 0.0

    drift = float(allee_drift(n, r, K, A))
    baseline = turnover * n
    birth = baseline + max(drift, 0.0)
    death = baseline + max(-drift, 0.0)

    if max_population is not None and n >= max_population:
        birth = 0.0

    return birth, death


def simulate_allee_gillespie(n0, r, K, A, t_max, rng, turnover=0.2, max_population=None, max_steps=100000):
    """Simulate one stochastic population path with Allee effect."""
    t = 0.0
    n = int(n0)
    times = [t]
    values = [n]

    for _ in range(max_steps):
        if t >= t_max or n <= 0:
            break

        birth, death = allee_birth_death_rates(n, r, K, A, turnover, max_population)
        total_rate = birth + death
        if total_rate <= 0:
            break

        tau = -math.log(max(rng.random(), np.finfo(float).tiny)) / total_rate
        if t + tau > t_max:
            break

        if rng.random() * total_rate < birth:
            n += 1
        else:
            n = max(n - 1, 0)

        t += tau
        times.append(t)
        values.append(n)

    if times[-1] < t_max:
        times.append(t_max)
        values.append(values[-1])

    return np.array(times), np.array(values)


def integrate_allee_mean_field(n0, t_grid, r, K, A):
    """Euler integration of the deterministic Allee mean-field equation."""
    values = np.empty_like(t_grid, dtype=float)
    values[0] = float(n0)
    for i in range(1, t_grid.size):
        dt = t_grid[i] - t_grid[i - 1]
        values[i] = max(0.0, values[i - 1] + dt * float(allee_drift(values[i - 1], r, K, A)))
    return values


allee_r = 0.45
allee_K = 80
allee_A = 25
allee_turnover = 0.25
allee_t_max = 55.0
allee_grid = np.linspace(0, allee_t_max, 300)
allee_initials = [12, 25, 42]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for n0 in allee_initials:
    for rep in range(5):
        times, values = simulate_allee_gillespie(
            n0,
            allee_r,
            allee_K,
            allee_A,
            allee_t_max,
            rng,
            turnover=allee_turnover,
            max_population=2 * allee_K,
        )
        label = f"stochastic N0={n0}" if rep == 0 else None
        axes[0].step(times, values, where="post", alpha=0.35, label=label)

    mean_field = integrate_allee_mean_field(n0, allee_grid, allee_r, allee_K, allee_A)
    axes[0].plot(allee_grid, mean_field, linewidth=2.5, label=f"mean field N0={n0}")

axes[0].axhline(allee_A, color="black", linestyle=":", label="Allee threshold A")
axes[0].axhline(allee_K, color="gray", linestyle="--", label="carrying capacity K")
axes[0].set(title="Stochastic Paths vs Mean Field", xlabel="time", ylabel="population N")
axes[0].legend(fontsize=8, ncol=2)

for n0 in [18, 25, 32]:
    final_values = []
    extinct = 0
    for _ in range(80):
        _, values = simulate_allee_gillespie(
            n0,
            allee_r,
            allee_K,
            allee_A,
            allee_t_max,
            rng,
            turnover=allee_turnover,
            max_population=2 * allee_K,
        )
        final_values.append(values[-1])
        extinct += values[-1] == 0
    final_values = np.array(final_values)
    survival_probability = 1 - extinct / 80
    axes[1].hist(final_values, bins=np.arange(-0.5, 2 * allee_K + 1.5, 4), alpha=0.55, label=f"N0={n0}, survival={survival_probability:.2f}")

axes[1].axvline(allee_A, color="black", linestyle=":", label="A")
axes[1].axvline(allee_K, color="gray", linestyle="--", label="K")
axes[1].set(title="Extinction and Survival Outcomes", xlabel="final population", ylabel="replicates")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
population_grid = np.linspace(0, 120, 161)
threshold_grid = np.linspace(8, 50, 120)
drift_map = np.zeros((threshold_grid.size, population_grid.size))

for i, A_value in enumerate(threshold_grid):
    drift_map[i] = allee_drift(population_grid, allee_r, allee_K, A_value)

max_abs_drift = np.max(np.abs(drift_map))

x0_values = np.arange(0, 81, 8)
A_values = np.arange(10, 46, 5)
n_repeats = 18
t_max_heatmap = 45.0
extinction_probability = np.zeros((A_values.size, x0_values.size))

for i, A_value in enumerate(A_values):
    for j, n0 in enumerate(x0_values):
        extinct = 0
        for _ in range(n_repeats):
            _, values = simulate_allee_gillespie(
                n0,
                allee_r,
                allee_K,
                A_value,
                t_max_heatmap,
                rng,
                turnover=allee_turnover,
                max_population=2 * allee_K,
            )
            extinct += values[-1] == 0
        extinction_probability[i, j] = extinct / n_repeats

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

drift_image = axes[0].imshow(
    drift_map,
    origin="lower",
    aspect="auto",
    extent=[population_grid.min(), population_grid.max(), threshold_grid.min(), threshold_grid.max()],
    cmap="coolwarm",
    vmin=-max_abs_drift,
    vmax=max_abs_drift,
)
axes[0].plot(threshold_grid, threshold_grid, color="black", linestyle=":", linewidth=1.5, label="N = A")
axes[0].axvline(allee_K, color="gray", linestyle="--", linewidth=1.5, label="N = K")
axes[0].set(title="Mean-Field Drift Heatmap", xlabel="population N", ylabel="Allee threshold A")
axes[0].legend(fontsize=8)
plt.colorbar(drift_image, ax=axes[0], label="dN/dt")

ext_image = axes[1].imshow(
    extinction_probability,
    origin="lower",
    aspect="auto",
    extent=[x0_values.min(), x0_values.max(), A_values.min(), A_values.max()],
    cmap="magma",
    vmin=0,
    vmax=1,
)
axes[1].plot(A_values, A_values, color="white", linestyle=":", linewidth=1.5, label="N0 = A")
axes[1].set(title="Stochastic Extinction Probability", xlabel="initial population N0", ylabel="Allee threshold A")
axes[1].legend(fontsize=8)
plt.colorbar(ext_image, ax=axes[1], label="P(extinction by t_max)")

plt.tight_layout()
plt.show()

print("Rows: Allee threshold values", A_values)
print("Columns: initial population values", x0_values)
print("Extinction probability heatmap shape:", extinction_probability.shape)

**Exercise**

- `allee_A`를 키우면 stochastic trajectory의 extinction probability가 어떻게 변하는지 확인해 보세요.
- Heatmap에서 `N0 < A` 영역과 `N0 > A` 영역을 비교하고, stochastic survival이 mean-field prediction과 언제 달라지는지 설명해 보세요.

## 13. One simple fitting example

마지막으로 SciPy의 `curve_fit`을 사용해 birth-death model의 ensemble mean을 결정론적 ODE 해에 맞춥니다.
여기서는 simulation으로 만든 평균 곡선에서 `beta`와 `k`를 다시 추정합니다.

In [ ]:
from scipy.optimize import curve_fit

fit_beta_true = 9.0
fit_k_true = 0.45
fit_x0 = 3
fit_t_max = 18.0
fit_replicates = 250
fit_grid = np.linspace(0, fit_t_max, 90)
fit_reactions = np.array([[1], [-1]])


def fit_propensities(x):
    return np.array([fit_beta_true, fit_k_true * x[0]])


fit_ensemble = np.zeros((fit_replicates, fit_grid.size))
for i in range(fit_replicates):
    times, states, _ = simulate_gillespie(
        np.array([fit_x0]),
        fit_reactions,
        fit_propensities,
        fit_t_max,
        rng,
    )
    fit_ensemble[i] = sample_step_trajectory(times, states[:, 0], fit_grid)

fit_mean = np.mean(fit_ensemble, axis=0)


def birth_death_mean_model(t, beta, k):
    return beta / k + (fit_x0 - beta / k) * np.exp(-k * t)


initial_guess = [5.0, 0.3]
best_params, covariance = curve_fit(
    birth_death_mean_model,
    fit_grid,
    fit_mean,
    p0=initial_guess,
    bounds=([0.0, 1e-6], [np.inf, np.inf]),
    maxfev=10000,
)
fit_beta, fit_k = best_params
fit_stderr = np.sqrt(np.diag(covariance))

true_curve = birth_death_mean_model(fit_grid, fit_beta_true, fit_k_true)
fitted_curve = birth_death_mean_model(fit_grid, fit_beta, fit_k)

print(f"True beta: {fit_beta_true:.3f}, fitted beta: {fit_beta:.3f} +/- {fit_stderr[0]:.3f}")
print(f"True k: {fit_k_true:.3f}, fitted k: {fit_k:.3f} +/- {fit_stderr[1]:.3f}")
print(f"True steady state beta/k: {fit_beta_true / fit_k_true:.3f}")
print(f"Fitted steady state beta/k: {fit_beta / fit_k:.3f}")

plt.figure(figsize=(8, 4.5))
plt.plot(fit_grid, fit_mean, "o", markersize=4, alpha=0.7, label="ensemble mean")
plt.plot(fit_grid, true_curve, "--", linewidth=2, label="true ODE mean")
plt.plot(fit_grid, fitted_curve, linewidth=2, label="fitted ODE mean")
plt.title("Fitting Birth-Death Mean")
plt.xlabel("time")
plt.ylabel("mean X")
plt.legend()
plt.tight_layout()
plt.show()

**Try it**

- `fit_replicates`를 줄이면 fitted parameter가 얼마나 흔들리는지 확인해 보세요.
- `initial_guess`를 바꿔도 비슷한 결과로 수렴하는지 비교해 보세요.